# CMSC 173 &middot; Machine Learning &mdash; Week 10 Lab
## Classification: KNN, Naive Bayes & Decision Trees

Three classic classifiers, three completely different ideas: **KNN** (ask your neighbours),
**Naive Bayes** (use probability), and **Decision Trees** (ask yes/no questions). You'll build
the first two from scratch, watch their **decision regions**, and compute a tree's
**information gain** by hand.

**How this lab works.** Each part = a short **plain-English explainer**, a **code cell**
you run, a **line-by-line walkthrough** of what it did, and an **Answer here** box. The
code does the maths; we *graph* the results so you can see what is going on.

**NumPy + Matplotlib + scikit-learn (for the tree).** **Not graded.** About 60 minutes.

---
## Part 0 &middot; Setup + three clusters

Three blobs of points in 2-D &mdash; three classes to tell apart.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(173)

centres = np.array([[0,0], [3,3], [0,4]])
y = rng.integers(0, 3, 240)
X = rng.normal(centres[y], 0.9)

def show(ax): 
    for cls, mark in zip(range(3), 'o^s'):
        ax.scatter(X[y==cls,0], X[y==cls,1], marker=mark, alpha=0.6, label=f'class {cls}')
plt.figure(figsize=(6,5)); show(plt.gca()); plt.legend(); plt.title('three classes'); plt.tight_layout(); plt.show()

**Reading the code:** 240 points spread around three centres. `show()` is a helper we'll reuse to
draw the classes. The clusters overlap a little &mdash; enough to make the classifiers disagree.

---
## Part 1 &middot; KNN from scratch: ask your neighbours

K-Nearest-Neighbours has no training at all. To classify a new point: find the `k` closest points
in the data, and take a **majority vote** of their labels. That's the whole algorithm.

In [ ]:
def knn_predict(query, X, y, k=5):
    d = np.sqrt(((X - query)**2).sum(axis=1))         # (1) distance to every point
    nearest = np.argsort(d)[:k]                        # (2) indices of the k closest
    votes = np.bincount(y[nearest], minlength=3)       # (3) tally their labels
    return votes.argmax()                              # (4) the winner

print('a point near class 1 is predicted:', knn_predict(np.array([3, 3]), X, y, k=5))
print('a point near class 0 is predicted:', knn_predict(np.array([0, 0]), X, y, k=5))

**Reading the code, line by line:**
- **(1)** Euclidean distance from the query point to *every* training point at once.
- **(2)** `argsort(d)[:k]` gives the indices of the `k` smallest distances &mdash; the neighbours.
- **(3)** `bincount` tallies how many of those neighbours belong to each class.
- **(4)** `argmax` returns the class with the most votes. A point sitting on a cluster is predicted
  as that cluster &mdash; exactly as you'd hope.

---
## Part 2 &middot; Decision regions, and the effect of k

Colour *every* point in the plane by what KNN would predict there &mdash; that's the **decision region**.
Small `k` gives jagged, noise-chasing regions (overfit); large `k` gives smooth ones. Compare.

In [ ]:
xx, yy = np.meshgrid(np.linspace(-3,6,120), np.linspace(-3,7,120))
grid = np.column_stack([xx.ravel(), yy.ravel()])

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, k in zip(ax, [1, 25]):
    preds = np.array([knn_predict(g, X, y, k) for g in grid]).reshape(xx.shape)
    a.contourf(xx, yy, preds, alpha=0.3, cmap='viridis')
    show(a); a.set_title(f'KNN decision regions, k={k}')
plt.tight_layout(); plt.show()

**Reading the code:** we run `knn_predict` on a dense grid and shade by the result. With `k=1` the
boundary is ragged &mdash; it bends around single stray points (**high variance / overfitting**). With
`k=25` it's smooth and general. Same lesson as choosing lambda or the divisor: complexity you can
dial up or down.

**Answer here:**

1. Which `k` would you trust more on *new* data, and why? Connect it to bias vs variance.
   &rarr; *your answer*

---
## Part 3 &middot; Naive Bayes from scratch: use probability

Naive Bayes flips the question: for each class, how *probable* is this point? It learns each class's
mean and variance, then uses Bayes' rule. 'Naive' = it pretends the features are independent, which
makes the maths a product of simple bell curves.

In [ ]:
def gaussian(x, mu, var):
    return np.exp(-(x-mu)**2 / (2*var)) / np.sqrt(2*np.pi*var)

# 'train': per-class prior, mean, variance for each feature
priors = np.array([np.mean(y==c) for c in range(3)])
means  = np.array([X[y==c].mean(axis=0) for c in range(3)])
varis  = np.array([X[y==c].var(axis=0)  for c in range(3)])

def nb_predict(query):
    # score each class = prior * product of per-feature bell curves
    scores = priors * np.prod(gaussian(query, means, varis), axis=1)
    return scores.argmax()

print('class priors:', np.round(priors, 2))
print('a point at (3,3) ->', nb_predict(np.array([3,3])))

**Reading the code:**
- `priors/means/varis` are the 'training': each class's share of the data, and its mean & variance
  per feature &mdash; three numbers-per-class, no iteration.
- `nb_predict` scores each class as *prior &times; how well the point fits that class's bell curves*,
  and picks the highest. That product-of-Gaussians *is* the naive-independence assumption in code.

In [ ]:
# decision regions for Naive Bayes, next to KNN(k=15)
fig, ax = plt.subplots(1, 2, figsize=(12,5))
nb_grid = np.array([nb_predict(g) for g in grid]).reshape(xx.shape)
ax[0].contourf(xx, yy, nb_grid, alpha=0.3, cmap='viridis'); show(ax[0]); ax[0].set_title('Naive Bayes')
kn_grid = np.array([knn_predict(g, X, y, 15) for g in grid]).reshape(xx.shape)
ax[1].contourf(xx, yy, kn_grid, alpha=0.3, cmap='viridis'); show(ax[1]); ax[1].set_title('KNN, k=15')
plt.tight_layout(); plt.show()

**Reading the code:** the two region maps look broadly similar here (the classes really are roughly
Gaussian blobs, which is Naive Bayes' happy case), but the *boundaries* differ &mdash; Naive Bayes draws
smooth curves from its bell-curve maths, KNN draws them from raw neighbour votes.

**Answer here:**

1. Naive Bayes 'trained' instantly (just means and variances); KNN does all its work at predict
   time. In one sentence each, name a situation favouring each.
   &rarr; *your answer*

---
## Part 4 &middot; Decision trees: information gain by hand

A tree asks yes/no questions to split the data into purer groups. **Entropy** measures impurity
(0 = one class only, high = mixed); **information gain** is how much a split *reduces* it. Let's
compute the gain of one candidate split, then let sklearn draw the full tree's regions.

In [ ]:
def entropy(labels):
    if len(labels) == 0: return 0.0
    p = np.bincount(labels, minlength=3) / len(labels)
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

parent = entropy(y)
left  = y[X[:,0] < 1.5]                                # split on feature-0 < 1.5
right = y[X[:,0] >= 1.5]
weighted = (len(left)*entropy(left) + len(right)*entropy(right)) / len(y)
print(f'parent entropy        = {parent:.3f}')
print(f'after split (weighted)= {weighted:.3f}')
print(f'information gain       = {parent - weighted:.3f}')

**Reading the code:** `entropy` turns label counts into the impurity number. We split the data at
`feature-0 < 1.5`, measure the impurity of each side, average by size, and subtract from the parent
&mdash; that difference is the **information gain**. A tree just tries many such splits and greedily
keeps the one with the highest gain, over and over.

**Answer here:**

1. If a split produced two perfectly pure groups (each all one class), what would the weighted
   entropy be, and what would the information gain equal?
   &rarr; *your answer*

---
## Where you actually are

Set the pace honestly. Replace each `-` with: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| The KNN algorithm (in words) | - |
| Effect of k on the decision regions | - |
| How Naive Bayes scores a class | - |
| Entropy & information gain | - |
| Which classifier trains instantly vs at predict time | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one plain sentence: give one strength of KNN and one of Naive Bayes.**
&rarr; *your answer*

---
## Stretch &mdash; optional

Required part is done; nothing below is graded.

### Stretch &middot; A real decision tree

`from sklearn.tree import DecisionTreeClassifier` &mdash; fit one on `X, y`, print its `.score(X, y)`,
and (optional) plot its decision regions with the grid like above. Fill it in.

In [ ]:
# from sklearn.tree import DecisionTreeClassifier
# your code here: fit DecisionTreeClassifier(max_depth=4).fit(X, y); print its score


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 10

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/10/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 10 submission page](https://portal.latarak.com/course/cmsc173/lab/10/submit) and upload it.

Blank cells are fine and guesses are fine. Don't polish this until it hides what you knew.